In [ ]:
import altair as alt
from nltk.tokenize import sent_tokenize
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import umap

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, logging
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences, get_older_labelled_data
from dap_job_quality.getters.data_getters import load_s3_jsonl, save_to_s3
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.utils import prodigy_data_utils as pdu

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def match_spans_to_sentences(df):
    for idx, row in df.iterrows():
        span = row['span_sents']
        if isinstance(span, str):
            for sent in row['sentences']:
                if (span in sent) | (span==sent):
                    # logging.info(f"'''{span}''' is in {sent}")
                    df.at[idx, 'true_sentence'] = sent
        else:
            df.at[idx, 'true_sentence'] = None
    return df

def get_negative_example_sentences(df, id):
    subset = df[df['id'] == id]
    sentences = set(subset['sentences'].iloc[0])
    sentences_w_spans = set(subset['true_sentence'].unique())
    neg_sentences = sentences - sentences_w_spans
    return neg_sentences

def reduce_to_2D(vectors, random_state=1):
    """Helper function to reduce vectors to 2-d embeddings using UMAP, for visualisation purposes"""
    reducer = umap.UMAP(n_components=2, random_state=random_state)
    embedding = reducer.fit_transform(vectors)
    return embedding

# Concatenate older labelled data and the more recent batch that we did in May

In [ ]:
labelled_sents = get_labelled_job_sentences()

In [ ]:
len(labelled_sents)

In [ ]:
final_labelled_ids = set([sent['meta']['id'] for sent in labelled_sents])

In [ ]:
early_data = get_older_labelled_data()[10:]

In [ ]:
early_ids = set([record['id'] for record in early_data])

In [ ]:
# check that no data is duplicated
common_ids = final_labelled_ids.intersection(early_ids)
common_ids

In [ ]:
early_data_spans = pdu.get_spans_and_sentences(early_data, chunks=False)

In [ ]:
early_data_df = pd.DataFrame()

for id, val in early_data_spans.items():
    temp_df = pd.DataFrame(val)
    temp_df['id'] = id
    early_data_df = pd.concat([early_data_df, temp_df])
    
early_data_df = early_data_df.reset_index(drop=True)

early_data_df['span_sents'] = early_data_df['span'].apply(lambda x: sent_tokenize(x))
early_data_df = early_data_df.explode('span_sents')
early_data_df['sentences'] = early_data_df['text'].apply(lambda x: sent_tokenize(x))

early_data_df = early_data_df.reset_index(drop=True)

In [ ]:
early_data_df = match_spans_to_sentences(early_data_df)

In [ ]:
labelled_spans = pdu.get_spans_and_sentences(labelled_sents, chunks=True)

In [ ]:
labelled_spans_df = pd.DataFrame()

for id, val in labelled_spans.items():
    for chunk, nested_val in val.items():
        temp_df = pd.DataFrame(nested_val)
        temp_df['id'] = id
        temp_df['chunk'] = chunk
        labelled_spans_df = pd.concat([labelled_spans_df, temp_df])
    
labelled_spans_df = labelled_spans_df.reset_index(drop=True)

In [ ]:
labelled_spans_df['span_sents'] = labelled_spans_df['span'].apply(lambda x: sent_tokenize(x))
labelled_spans_df = labelled_spans_df.explode('span_sents')
labelled_spans_df['sentences'] = labelled_spans_df['text'].apply(lambda x: sent_tokenize(x))

labelled_spans_df = match_spans_to_sentences(labelled_spans_df)

In [ ]:
early_data_df = early_data_df[early_data_df["span"] != ""]
labelled_spans_df = labelled_spans_df[labelled_spans_df["label"] != "none"]

In [ ]:
all_labelled_data = pd.concat([early_data_df[['id', 'text', 'label','sentences', 'span_sents', 'true_sentence']], labelled_spans_df[['id', 'text', 'label','sentences', 'span_sents', 'true_sentence']]]).reset_index(drop=True)
all_labelled_data

In [ ]:
negative_sentences = {}

for id in all_labelled_data['id'].unique():
    neg_sentences = get_negative_example_sentences(all_labelled_data, id)
    negative_sentences[id] = list(neg_sentences)

In [ ]:
neg_sent_df = pd.DataFrame([(k, sentence) for k, sentences in negative_sentences.items() for sentence in sentences], columns=['id', 'sentence'])
neg_sent_df

In [ ]:
# hard-coding false negatives here, rather than removing manually - just so that there's a record of what was removed
false_negatives = ["Basic entitlement is 30.0 days (pro rata for hours worked).",
                   "Location  Bromley.",
                   "We also offer an i. Pad if you refer a new client to us and we recruit for them.",
                   "Overtime rates",
                   "7 days on call",
                   "Monday start of shift to handover the following Monday start of shift.",
                   "You will be given a training program when you start to help you integrate into the team and help you get up to date with their technologies as well as progression routes.",
                   "Work between 8.30am and 3.30pm",
                   "Salary £22,000 - 26,000p a",
                   "£250 bonus",
                   "You can discuss your preferred working hours options at interview.",
                   "A full shift would be 9-5 but there are variations of hours on offer.",
                   "Recommend a friend",
                   "6-9 calls per day dependant on area size.",
                   "They have just moved to some fantastic brand-new offices too based in South Cerney.",
                   "We are also proud to have been named a 'Top Employer' for 5 consecutive years.",
                   "You can earn up to £31,500 p a including regular overtime and bonuses.",
                   "Salary  £50,000 - £60,000",
                   "You will receive £250 for every candidate we place in permanent employment who has been recommended by you.",
                   "relocation package",
                   "This gives Colleagues and their family access to 24 7 365 support for a whole range of issues including physical, mental and financial issues.",
                   "Refer a Friend",
                   "one of the best companies to work for",
                   "has topped the UK 'Best Companies to Work For' lists for 15 years",
                   "Join one of the world's best employers!",
                   "Our client are looking for a Security Cleared Pharmacy Technician to join their team on a locum basis starting as soon as possible on an ongoing basis.",
                   "monthly commission structure",
                   "with the chance of full-time job",
                   "4 Nights out a week and can sometimes have run ins on a Saturday.",
                   "Hours of work- 9-5Contract- temporary."
                   ]

In [ ]:
mask = neg_sent_df['sentence'].apply(lambda x: not any(sub in x for sub in false_negatives))

# Apply the mask to filter the DataFrame
neg_sent_df_filtered = neg_sent_df[mask]

In [ ]:
len(neg_sent_df_filtered)

In [ ]:
len(all_labelled_data)

# Clustering negative sentences

We will cluster the negative sentences and then sample randomly from each cluster, in order to ensure that the negative sample is fairly representative.

In [ ]:
embeddings = model.encode(neg_sent_df_filtered['sentence'].tolist())

embeddings_2d = reduce_to_2D(embeddings)

num_clusters = 50

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(embeddings_2d)

In [ ]:
# assign the cluster names back into the dataframe
neg_sent_df_filtered['cluster'] = clusters

neg_sent_df_filtered['cluster'].value_counts()

neg_sent_df_filtered['x'] = embeddings_2d[:, 0]
neg_sent_df_filtered['y'] = embeddings_2d[:, 1]

Visualise the clusters to check that they seem sensible:

In [ ]:
# Create the Altair chart
chart = alt.Chart(neg_sent_df_filtered).mark_circle(size=60).encode(
    x='x:Q',
    y='y:Q',
    color='cluster:N',
    tooltip=['cluster','sentence']
).properties(width=900, height=600).interactive()

# Display the chart
chart.show()

In [ ]:
chart.save(PROJECT_DIR / 'outputs/figures/negative_sentences_clusters.html')

In [ ]:
sample_size = round(len(all_labelled_data) / num_clusters)

In [ ]:
neg_sent_df_sample = neg_sent_df_filtered.groupby('cluster', group_keys=False).apply(lambda x: x.sample(min(len(x), sample_size)))

In [ ]:
len(neg_sent_df_sample)

In [ ]:
neg_sent_df_sample['label'] = 0
neg_sent_df_sample['span_sents'] = None

In [ ]:
all_labelled_data['label'] = 1

In [ ]:
all_labelled_data.rename(columns={'true_sentence': 'sentence'}, inplace=True)
all_labelled_data

We save the positive sentences and manually label them here:
https://docs.google.com/spreadsheets/d/1bXNmO9vOLG6zdDpHl0Tdw9AeDqb43CrpkXzNGyHRhWI/edit?gid=4769099#gid=4769099

In [ ]:
save_to_s3(BUCKET_NAME,
           all_labelled_data,
           'job_quality/sentence_classifier/inputs/labelling/positive_sents_for_labelling.csv')

# Create a train/ validation/ test split

In [ ]:
# Concatenate the positive and negative examples
all_data = pd.concat([neg_sent_df_sample[['id', 'sentence', 'label', 'span_sents']],all_labelled_data[['id', 'sentence', 'label', 'span_sents']]]).reset_index(drop=True)

In [ ]:
# Group by 'id' and calculate the proportion of each label within each 'id'
id_labels = all_data.groupby('id')['label'].apply(lambda x: x.value_counts(normalize=True)).unstack(fill_value=0)

# Add a column with the majority label for stratification
id_labels['majority_label'] = id_labels.idxmax(axis=1)

# Get unique ids and their majority label
ids_with_labels = id_labels.reset_index()[['id', 'majority_label']]

# Split the unique ids into training, validation, and test sets using stratified sampling
train_ids, temp_ids = train_test_split(ids_with_labels, test_size=0.3, stratify=ids_with_labels['majority_label'], random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, stratify=temp_ids['majority_label'], random_state=42)

In [ ]:
# Create the training, validation, and test DataFrames
train_df = all_data[all_data['id'].isin(train_ids['id'])]
print(f"Train size: {len(train_df)}")
logging.info(train_df['label'].value_counts())
val_df = all_data[all_data['id'].isin(val_ids['id'])]
print(f"Val size: {len(val_df)}")
logging.info(val_df['label'].value_counts())
test_df = all_data[all_data['id'].isin(test_ids['id'])]
print(f"Test size: {len(test_df)}")
logging.info(test_df['label'].value_counts())

save_to_s3(BUCKET_NAME, train_ids, 'job_quality/sentence_classifier/inputs/labelled/train_ids.parquet')
save_to_s3(BUCKET_NAME, val_ids, 'job_quality/sentence_classifier/inputs/labelled/val_ids.parquet')
save_to_s3(BUCKET_NAME, test_ids, 'job_quality/sentence_classifier/inputs/labelled/test_ids.parquet')

save_to_s3(BUCKET_NAME, train_df, 'job_quality/sentence_classifier/inputs/labelled/train_df.parquet')
save_to_s3(BUCKET_NAME, val_df, 'job_quality/sentence_classifier/inputs/labelled/val_df.parquet')
save_to_s3(BUCKET_NAME, test_df, 'job_quality/sentence_classifier/inputs/labelled/test_df.parquet')

# EDA of categories in labelled data

Now we load back in the data that was labelled here:
https://docs.google.com/spreadsheets/d/1bXNmO9vOLG6zdDpHl0Tdw9AeDqb43CrpkXzNGyHRhWI/edit?gid=4769099#gid=4769099

In [ ]:
positive_sents_manually_labelled = pd.read_csv(
        "s3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/positive_sents_for_labelling - positive_sents_for_labelling.csv"
    )

In [ ]:
category_representation = pd.DataFrame(positive_sents_manually_labelled['subcategory'].value_counts(normalize=True)).reset_index()
category_representation['perc'] = round(category_representation['proportion'] * 100)

In [ ]:
keywords = get_keywords()

In [ ]:
category_representation = pd.merge(category_representation, keywords[['dimension','subcategory']].drop_duplicates(), on='subcategory', how='outer')
category_representation

In [ ]:
category_representation.to_csv(PROJECT_DIR / "outputs/data/sent_classifier_category_representation.csv", index=False)